<a href="https://colab.research.google.com/github/edi-kanyua/Data-Anonymization/blob/main/Data_Anonymization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bank Customer Data Anonymization

> ## Project Overview
This notebook demonstrates data anonymization techniques applied to a dataset containing bank customer personal and financial info. The objective is to reduce the risk of exposing personally identifiable info and sensitive customer data while retaining optimum analytical value of the data.
This notebook applies the following anonymization techniques depending on the sensitivity of each attribute:
  - Consistent date shifting
  - Suppression of direct identifiers
  - Data masking
  - Pseudonymization using Faker
  - Generalization
  - Tokenization

> **Privacy Note:** The original dataset contains sensitive customer data. Real customer info have not been uploaded or publicly distributed; rather, any publicly public shared data of this project is synthetic, appropriately anonymized, or authorized.

> ## Objectives
*   Identify direct identifiers, quasi-identifiers, and sensitive attributes within customer data.
*   Apply appropriate anonymization techniques to different types of sensitive information.
*   Preserve useful relationships and analytical characteristics where possible.

> ## Anonymization Techniques
**Consistent Date Shifting:** Changes dates by a consistent offset rather than changing individual dates independently so as to preserve relationships between the dates; applied to date_registered and birthdate cols.

> **Suppressing Direct Identifiers:** Direct identifiers are removed from the dataset; applied to customer_id and current_location cols.

> **Data Masking:** Partially obscures sensitive info while retaining a limited portion of the original data; applied to email, username, credit_card_number, and credit_card_security_code cols.

> **Pseudonymization with Faker:** The Faker library is used to replace identifying info with synthetic values; applied to name, address, residence cols.

> **Generalization:** Reduces the precision of sensitive numerical attributes; age and salary cols are converted into age groups and salary bands respecitvely.

> **Tokenization:** Replaces sensitive info with generated tokens; applied to credit_card_provider and credit_card_expire cols.




















## Import Libraries, Setup, and Load Dataset

In [1]:
# Connect to drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.5 MB/s eta 0:00:00


In [3]:
# Import libraries
import pandas as pd # data manipulation and transformation
import numpy as np # numerical operations
import re # pattern-based masking
import hashlib # cryptographic hashing

from faker import Faker # generation of synthetic values

import matplotlib.pyplot as plt # visualizations
import seaborn as sns
sns.set()

In [4]:
# Load dataset
df = pd.read_excel("/content/drive/MyDrive/Common Wealth Bank/Mobile_customers_raw.xlsx", index_col=0)

## Data Overview & Inspection

In [ ]:
# Inspect dataset to understand structure, data types and attributes with sensitive info.
print(f"df shape:\n{df.shape}\n")

print("df info:")
df.info()

print("\nSummary Statistics:")
df.describe()

df shape:
(10000, 18)

df info:
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   customer_id                10000 non-null  object        
 1   date_registered            10000 non-null  datetime64[ns]
 2   username                   10000 non-null  object        
 3   name                       10000 non-null  object        
 4   gender                     10000 non-null  object        
 5   address                    10000 non-null  object        
 6   email                      10000 non-null  object        
 7   birthdate                  10000 non-null  datetime64[ns]
 8   current_location           10000 non-null  object        
 9   residence                  10000 non-null  object        
 10  employer                   10000 non-null  object        
 11  job                        10000 non-null

,date_registered,birthdate,age,salary,credit_card_number,credit_card_security_code
count,10000,10000,10000.000000,10000.000000,1.000000e+04,10000.00000
mean,2020-12-21 07:08:58.560000,1964-10-01 19:25:58.079999968,41.618000,133657.802800,3.669000e+17,856.19800
min,2019-06-22 00:00:00,1906-06-28 00:00:00,18.000000,20014.000000,6.040106e+10,0.00000
25%,2020-03-21 00:00:00,1935-09-11 06:00:00,30.000000,77959.500000,1.800136e+14,270.00000
50%,2020-12-23 12:00:00,1964-08-28 00:00:00,42.000000,133583.500000,3.513585e+15,534.50000
75%,2021-09-17 00:00:00,1994-04-13 06:00:00,54.000000,189975.500000,4.595706e+15,804.00000
max,2022-06-21 00:00:00,2022-06-07 00:00:00,65.000000,244926.000000,4.997334e+18,9996.00000
std,NaN,NaN,13.858812,64635.054802,1.230507e+18,1485.41048


## Data Anonymization Pipeline

In [5]:
"""
    Anonymize the customer dataset using a combination of:
        - Consistent Date Shifting
        - Masking
        - Suppression of direct identifiers
        - Pseudonymization (Faker)
        - Generalization
        - Tokenization
  """

def anonymize_dataframe(df):

    # Create a copy of original df

    df = df.copy() # handy during validation stage

    # Instantiate Faker object

    fake = Faker()
    fake.seed_instance(18) # control randomization

    # Mapping dictionaries

    name_map = {}
    address_map = {}
    residence_map = {}
    provider_map = {}
    expire_map = {}

    # Helper functions

    def date_offset(identifier, max_days=30):
        h = hashlib.sha256(str(identifier).encode()).hexdigest()
        value = int(h[:8], 16)
        return (value % (2 * max_days + 1)) - max_days

    def mask_email(email):
        if pd.isna(email):
            return email

        match = re.match(r"(.).*(@.*)", str(email))

        if match:
            return f"{match.group(1)}****{match.group(2)}"

        return email

    def mask_username(username):
        if pd.isna(username):
            return username

        def mask(word):
            if len(word) <= 2:
                return word

            return word[0] + "*" * (len(word) - 2) + word[-1]

        return " ".join(mask(w) for w in str(username).split())

    def fake_mapper(value, mapping, generator):
        if pd.isna(value):
            return value

        if value not in mapping:
            mapping[value] = generator()

        return mapping[value]

    def tokenize(value, mapping, prefix, digits):
        if pd.isna(value):
            return value

        if value not in mapping:
            mapping[value] = f"{prefix}{len(mapping)+1:0{digits}d}"

        return mapping[value]

    def mask_card_number(number):
        if pd.isna(number):
            return number

        number = str(number)
        return "*" * (len(number) - 4) + number[-4:]

    def mask_cvv(code):
        if pd.isna(code):
            return code

        code = str(code)
        return "*" * len(code)

    # Consistent Date Shifting

    offsets = df["customer_id"].apply(date_offset)

    # Ensure dates are in datetime format

    df["birthdate"] = pd.to_datetime(df["birthdate"])
    df["date_registered"] = pd.to_datetime(df["date_registered"])

    # Apply offset

    df["birthdate"] += pd.to_timedelta(offsets, unit="D")
    df["date_registered"] += pd.to_timedelta(offsets, unit="D")

    # Suppress direct identifiers

    df.drop(columns=["customer_id", "current_location"], inplace=True)

    # Mask fields

    df["email"] = df["email"].map(mask_email)
    df["username"] = df["username"].map(mask_username)

    # Convert to string before masking
    df["credit_card_number"] = df["credit_card_number"].astype("string")
    df["credit_card_security_code"] = df["credit_card_security_code"].astype("string")

    df["credit_card_number"] = df["credit_card_number"].map(mask_card_number)
    df["credit_card_security_code"] = df["credit_card_security_code"].map(mask_cvv)

    # Pseudonymize with Faker

    df["name"] = df["name"].map(
        lambda x: fake_mapper(x, name_map, fake.name)
    )

    df["address"] = df["address"].map(
        lambda x: fake_mapper(x, address_map, fake.address)
    )

    df["residence"] = df["residence"].map(
        lambda x: fake_mapper(x, residence_map, fake.address)
    )

    # Generalize

    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 29, 45, 61, float("inf")],
        labels=["18–29", "30–44", "45–60", "61+"],
        right=False,
    )

    df["salary_band"] = pd.cut(
        df["salary"],
        bins=[0, 30000, 60000, 100000, 200000, float("inf")],
        labels=["0–30K", "30–60K", "60–100K", "100–200K", "200K+"],
        right=False,
    )

    df.drop(columns=["age", "salary"], inplace=True)

    # Tokenize

    df["credit_card_provider"] = df["credit_card_provider"].map(
        lambda x: tokenize(x, provider_map, "CP", 2)
    )

    df["credit_card_expire"] = df["credit_card_expire"].map(
        lambda x: tokenize(x, expire_map, "T", 5)
    )

    return df

df_anon = anonymize_dataframe(df)

df_anon.head()

,date_registered,username,name,gender,address,email,birthdate,residence,employer,job,credit_card_provider,credit_card_number,credit_card_security_code,credit_card_expire,age_group,salary_band
0,2021-09-23,r**********n,Matthew Cook,M,"159 Nancy Camp\nDeannamouth, CT 63996",m****@hotmail.com,1978-03-05,"234 Evans Trace\nMichaelburgh, SC 98104","Byrd, Welch and Holt",Chief Technology Officer,CP01,**********9846,***,T00001,45–60,30–60K
1,2019-07-20,e*****a,John Snyder,F,"9704 Taylor Parks Suite 300\nMaryton, ME 09586",a****@hotmail.com,1970-11-01,"720 Wood Spurs\nRobertschester, PA 46779",Hurst PLC,Data scientist,CP02,************5979,***,T00002,30–44,60–100K
2,2019-11-19,t*********n,Jimmy Baker,M,"4105 Simmons Rapid\nStevehaven, MA 78800",v****@gmail.com,2009-05-11,"765 James Lock Apt. 336\nNew Amanda, WV 34012","Mora, Caldwell and Guerrero",Chief Operating Officer,CP03,***************2247,***,T00003,45–60,200K+
3,2022-01-11,r*************l,Michael Nguyen,F,"6786 Nicholson Rapids\nLake Thomasshire, MO 09741",k****@gmail.com,1992-08-07,01250 Castaneda Trace Suite 099\nWest Josephvi...,Patel PLC,Counselling psychologist,CP03,***************7844,****,T00004,30–44,100–200K
4,2020-07-23,t************n,Rachel Cline,F,"63277 Crystal Crescent\nJohnview, MN 40679",j****@hotmail.com,1989-08-30,"25914 Pratt Gardens Suite 779\nMichaelside, AZ...",Smith-Mejia,Mining engineer,CP04,***********8217,**,T00005,45–60,100–200K


## Validation

> After applying the anonymization tranformations, the resulting data should be validated to ensure that the intended privacy protections were applied correctly; the validation stage also checks whether there was unnecessary data loss or corruption.



In [41]:
# Check whether records have been accidentally removed or added
print("Original Dataset Shape: ", df.shape)
print("Anonymized Dataset Shape: ", df_anon.shape)

print("All rows preserved?", df.shape[0] == df_anon.shape[0])

# Validate suppression of direct identifiers
suppressed_cols = [
    'customer_id',
    'current_location',
    'age',
    'salary'
]

print("\nWere suppressed cols removed successfully?")
for col in suppressed_cols:
  print(f"{col}: {col not in df_anon.columns}")

# Validate masking credit card details
card_validate = df_anon['credit_card_number'].dropna().astype(str).str.match(r"^\*+\d{4}$")
print("\nCredit card number successfully masked?", card_validate.all())

cvv_validate = df_anon['credit_card_security_code'].dropna().astype(str).str.match(r"^\*+$")
print("CVV successefully masked?", cvv_validate.all())

# Validate pseudonymization consistency: no. of anonymized names per original name
name_consistency = (
    pd.DataFrame(
        {
            "Original": df['name'],
            "Anonymized": df_anon['name']
        }
    ).groupby("Original")["Anonymized"]
    .nunique()
    .max()
)

print("\nMax anonymized names per original name: ", name_consistency)

# Do the same for address
address_consistency = (
    pd.DataFrame(
        {
            "Original": df['address'],
            "Anonymized": df_anon['address']
        }
    ).groupby("Original")["Anonymized"]
    .nunique()
    .max()
)

print("Max anonymized addresses per original address: ", address_consistency)

Original Dataset Shape:  (10000, 18)
Anonymized Dataset Shape:  (10000, 16)
All rows preserved? True

Were suppressed cols removed successfully?
customer_id: True
current_location: True
age: True
salary: True

Credit card number successfully masked? True
CVV successefully masked? True

Max anonymized names per original name:  1
Max anonymized addresses per original address:  1


Max anaonymized names per original name:  1


In [20]:
df['credit_card_number'].head()

,credit_card_number
0,38985874269846
1,6525743622515979
2,4010729915028682247
3,4854862659569207844
4,213152724828217
